# Imports

In [ ]:
!pip install google-generativeai

In [ ]:
import google.generativeai as genai
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

# Prompts

In [ ]:
pre_prompt_inicial = """
This pre-prompt only sets role and parsing rules;
do NOT produce final output here.
delimiters = <pre_prompt> </pre_prompt>

<pre_prompt>
  ROLE: You are an expert data parser for XAI (SHAP/LIME/IG) and BERT outputs and JSON structure analizer. your task is...

  CORE RULES:
  1. NO CONVERSATION: Just answer the questions specified by the Answer token: nothing else.
  2. DATA EXTRACTION: Evaluate the JSON input values ​​to answer the question fields. If a specific key, such as "explanation," is mentioned in the instructions, prioritize it as the source of truth.
  3. STEP DETECTION: Only output "No steps provided" if the prompt explicitly asks for a step-by-step extraction and none is found.
  4. Look at the prompt type within the prompt in the `prompt_type` key and adapt to perform in the best way for each prompt type.
  5. Your primary source of information about the sentiment of the text comes from BERT's explanations and the XAI (shape, lime, integrated gradients) analysis algorithms. When evaluating, use these indicators as the primary source of truth.
  6. Your job is not to observe the sentence itself. Observe what BERT and its evaluation algorithms say about it. Don't try to guess which word or phrase might have influenced BERT's response; if that information is provided by BERT or one of its supervisory algorithms, then you can include it.
  7. You must act exclusively as an Extraction Parser. It is strictly forbidden to infer, deduce, or create information that is not present in the JSON keys. If the command requires synthesis, use only the words contained in the input.
  8. Before answering any of the questions in the questions block, rewrite the question being answered in the output.
  9. It is strictly forbidden to generate an answer to a question with more than 3 lines of response.
  10. In all outputs, display the first part (roughly the first 30 tokens) of the analyzed sentence in the sentence pattern: (first part of the analyzed sentence).
</pre_prompt>
"""

# Model prompts


In [ ]:
zero_shot_prompt1 = """
<prompt>
  Prompt_type = zero_shot

  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Did the author highlight more positive or negative points?

</prompt>
"""

In [ ]:
few_shot_prompt1 = """
<prompt>
  Prompt_type = few_shot

  ### BEHAVIOR EXAMPLES ###

  Observe the questions in advance:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Did the author highlight more positive or negative points?

  Example 1 (Positive)
  JSON Input: {"explanations": {"sentiment": "POSITIVO", "confidence": 0.9991152882575989}, "probabilities": [0.0008846964919939637, 0.9991152882575989], "sentence": ["Petter Mattei's Love in the Time of Money is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situations we encounter.   This being a variation on the Arthur Schnitzler's play about the same theme, the director transfers the action to the present time New York where all these different characters meet and connect. Each one is connected in one way, or another to the next person, but no one seems to know the previous point of contact. Stylishly, the film has a sophisticated luxurious look. We are taken to see how these people live and the world they live in their own habitat.  The only thing one gets out of all these souls in the picture is the different stages of loneliness each one inhabits. A big city is not exactly the best place in which human relations find sincere fulfillment, as one discerns is the case with most of the people we encounter.  The acting is good under Mr. Mattei's direction. Steve Buscemi, Rosario Dawson, Carol Kane, Michael Imperioli, Adrian Grenier, and the rest of the talented cast, make these characters come alive.  We wish Mr. Mattei good luck and await anxiously for his next work."]}

  Expected output:
  The Bert AI concluded that the tone of this comment is POSITIVO (Positive). The specific reasons (e.g., influential words) for this conclusion are not provided by the BERT explanation.
  Bert is 99.91% certain about this conclusion.
  This information was not provided.

  Example 2 (Positive - Replicable)
  JSON input : {"explanations": {"sentiment": "POSITIVO", "confidence": 0.999421501159668},"probabilities": [0.0005784988403320312, 0.999421501159668],"sentence": ["An absolute adrenaline rush from start to finish! The stunts are breathtaking and the choreography is world-class. If you love action, this is the definitive movie of the year. The lead actor delivers a powerhouse performance that keeps you on the edge of your seat. Truly a theater experience you cannot miss."]}

  Expected output:
  The Bert AI concluded that the tone of this comment is POSITIVO (Positive). The specific reasons (e.g., influential words) for this conclusion are not provided by the BERT explanation.
  Bert is 99.94% certain about this conclusion.
  This information was not provided.


  Example 3 (Positive - Replicable)
  JSON input : {"explanations": {"sentiment": "POSITIVO", "confidence": 0.9887321453094482},"probabilities": [0.011267854690551758, 0.9887321453094482],"sentence": ["A deeply moving and thought-provoking science fiction piece. It explores human nature in a way that few films dare to. The cinematography is hauntingly beautiful, and the score perfectly complements the somber yet hopeful tone of the narrative. A modern classic in the making."]}

  Expected output:
  The Bert AI concluded that the tone of this comment is POSITIVO (Positive). The specific reasons (e.g., influential words) for this conclusion are not provided by the BERT explanation.
  Bert is 98.87% certain about this conclusion.
  This information was not provided.

  Example 4 (Negative - Non-Replicable)
  Reason for Non-Replication: The AI interprets implicit context from the text (acting, boredom) that is not explicitly detailed in the BERT metadata fields. JSON Input:

  JSON input : {"explanations": {"sentiment": "NEGATIVO", "confidence": 0.9996542930603027},"probabilities": [0.9996542930603027, 0.0003457069396972656],"sentence": ["I have never been so bored in a cinema. The acting was wooden and the script felt like it was written by an amateur. I wanted to leave after the first twenty minutes. Total disaster."]}

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? The comment is classified as negative. This is justified by the author's use of phrases like "wooden acting" and "total disaster," suggesting a failure in the film's production.
  How certain is the Bert about this? There is 99.97% confidence in this analysis.
  Did the author highlight more positive or negative points? The author highlighted only negative points.

  Example 5 (Negative - Non-Replicable)
  Reason for Non-Replication: The AI provides a detailed summary and qualitative evaluation not present in the JSON explanation object. JSON Input:

  JSON input : {"explanations": {"sentiment": "NEGATIVO", "confidence": 0.9754123687744141},"probabilities": [0.9754123687744141, 0.024587631225585938],"sentence": ["The sequel fails to capture any of the magic of the original. It is overlong, messy, and the special effects look cheap. Avoid this one at all costs."]}
  Expected output:

  Bert concluded the tone is negative because the author describes the film as "cheap" and "messy."
  Bert is 97.54% certain.
  The author highlighted more negative points.

  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Did the author highlight more positive or negative points?

</prompt>
"""


In [ ]:
chain_of_thought_prompt1 = """
<prompt>
  Prompt_type = chain_of_thought

  ANALYZE THROUGH STRUCTURED REASONING:

  STEP 1 - DATA MAPPING:
  • Extract the "sentence" from the JSON.
  • Identify the "sentiment" and "confidence" values in the "explanations" field.
  • Convert the "confidence" decimal (0-1) to a 0-10 scale and a percentage (e.g., 0.99 -> 9.9/10 or 99%).

  STEP 2 - CONSISTENCY EVALUATION:
  • Apply the Consistency Rules:
    - Value >= 8.0: High
    - Value 5.0 - 7.9: Medium
    - Value < 5.0: Low
    Justify your answer based on the data from BERT.
  • Determine the reliability of the AI's conclusion based on these intervals.

  STEP 3 - TEXTUAL EVIDENCE EXTRACTION:
  • Search for specific mentions of a numerical score (e.g., "X out of 10"). Use this on the future explanations.

  STEP 4 - LAYPERSON SYNTHESIS:
  • Synthesize the findings into a narrative that avoids technical jargon (no "JSON", "BERT", or "float").
  • Prepare the answers for the 5 final questions.
  • Answer the questions. Avoid using technical language such as JSON and tags. Provide user-friendly answers for non-experts.

  STEP 5 - Instructions on how to answer the questions.
  • Do not generate explanations for data not provided by the JSON. If information about the author's intent is not in the JSON, do not include it in the explanations.
  • Don't give lengthy explanations for the questions. Try to use few words but be concise in your explanation.
  • For each answer, use only 1.5 lines to explain.
  • Standardize the output using user-friendly indentation and rewrite the question in the answer, generating an additional 1.5 lines of response.
  • Do not add extra information to the explanations; answer only and strictly what was requested in the question.
  • If the requested information is not explicitly contained in the input's JSON, simply respond: This information was not provided.
  • Remove all types of markers that are not human-friendly, such as: '### ###', '** **', and all tags like <tag>, <start>, <end>.
  -------------------------------------------------------------------------------
  TASK: Based on the steps above, answer the following questions for the provided JSON. Include the obtained values ​​and justifications in your answers.

  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Did the author highlight more positive or negative points?
</prompt>
"""

# XAI output prompts


In [ ]:
zero_shot_prompt2 = """
<prompt>
  Prompt_type = zero_shot

  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Which XAI algorithms were evaluated?
  4. What do the XAI algorithms say about the phrase and the answer given by Bert?

</prompt>
"""

In [ ]:
few_shot_prompt2 = """
<prompt>
  Prompt_type = few_shot_prompt

  Example 1 (Positive)
  JSON Input:{
  "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
  "model": "distilbert-base-uncased-finetuned-sst-2-english",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "xai_methods": [
    "SHAP",
    "LIME",
    "Integrated_Gradients"
  ],
  "sentences": [
    "Petter Mattei's Love in the Time of Money is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situations we encounter.   This being a variation on the Arthur Schnitzler's play about the same theme, the director transfers the action to the present time New York where all these different characters meet and connect. Each one is connected in one way, or another to the next person, but no one seems to know the previous point of contact. Stylishly, the film has a sophisticated luxurious look. We are taken to see how these people live and the world they live in their own habitat.  The only thing one gets out of all these souls in the picture is the different stages of loneliness each one inhabits. A big city is not exactly the best place in which human relations find sincere fulfillment, as one discerns is the case with most of the people we encounter.  The acting is good under Mr. Mattei's direction. Steve Buscemi, Rosario Dawson, Carol Kane, Michael Imperioli, Adrian Grenier, and the rest of the talented cast, make these characters come alive.  We wish Mr. Mattei good luck and await anxiously for his next work."
  ],
  "explanations": {
    "bert": {
      "sentiment": "POSITIVO",
      "confidence": 0.9991152882575989,
      "probabilities": [
        0.0008846964919939637,
        0.9991152882575989
      ]
    },
    "shap": {
      "shap_data_format": "Token: \"{token}\" -> {value:.4f}",
      "shap_data": [
        "<shap>",
        [
          "  Token: ' not' -> 0.0602",
          "  Token: ' look' -> -0.0352",
          "  Token: ' luxurious' -> -0.0352",
          "  Token: ' sophisticated' -> -0.0352",
          "  Token: ' .' -> -0.0320",
          "  Token: ' exactly' -> 0.0293",
          "  Token: ' alive' -> -0.0239",
          "  Token: ' come' -> -0.0239",
          "  Token: ' characters' -> -0.0239",
          "  Token: ' good' -> -0.0205",
          "  Token: ' under' -> -0.0187",
          "  Token: ' stunning' -> -0.0184",
          "  Token: ' visually' -> -0.0184",
          "  Token: ' .' -> -0.0180",
          "  Token: ' a' -> -0.0176",
          "  Token: ' vivid' -> -0.0176",
          "  Token: ' is' -> -0.0170",
          "  Token: ' make' -> -0.0170",
          "  Token: ' these' -> -0.0170",
          "  Token: ' a' -> 0.0144",
          "  Token: ' movie' -> 0.0144",
          "  Token: ' mr' -> -0.0143",
          "  Token: ' best' -> 0.0143",
          "  Token: ' about' -> -0.0141",
          "  Token: ' portrait' -> -0.0141",
          "  Token: ' .' -> -0.0135",
          "  Token: ' .' -> -0.0129",
          "  Token: ' direction' -> -0.0129",
          "  Token: ' the' -> 0.0126",
          "  Token: ' ,' -> -0.0121",
          "  Token: ' cast' -> -0.0121",
          "  Token: ' talented' -> -0.0121",
          "  Token: ' watch' -> -0.0115",
                        .
                        .
                        .
          "Positive_prob = 0.371878445148468"
        ],
        "</shap>"
      ]
    },
    "lime": {
      "lime_data_format": "Token: \"{token}\" -> {value:.4f}",
      "lime_data": [
        "<lime>",
        [
          [
            "0.029400 -> stunning",
            "-0.029127 -> not",
            "0.026575 -> good",
            "0.024989 -> vivid",
            "0.014425 -> Love",
            "0.014406 -> success",
            "0.014105 -> alive",
            "0.013706 -> look",
            "0.013552 -> Mattei",
            "-0.012130 -> no",
            "0.012039 -> film",
            "0.011570 -> portrait",
            "0.011442 -> action",
            "0.011077 -> human",
            "-0.010760 -> only",
            "-0.009538 -> the",
            "-0.007838 -> next",
            "-0.007804 -> Money",
            "-0.006511 -> thing",
            "-0.006007 -> anxiously"
          ]
        ],
        "</lime>"
      ]
    },
    "integrated_gradients": {
      "integrated_gradients_data_format": "Token: \"{token}\", \"importance\": {value:.4f}}",
      "integrated_gradients_data": [
        "<ig>",
        [
          {
            "token": "pet",
            "importance": 0.0008
          },
          {
            "token": "##ter",
            "importance": 0.004
          },
          {
            "token": "matt",
            "importance": 0.0026
          },
          {
            "token": "##ei",
            "importance": 0.0
          },
          {
            "token": "s",
            "importance": 0.0023
          },
          {
            "token": "love",
            "importance": 0.0041
          },
          {
            "token": "in",
            "importance": 0.0002
          },
          {
            "token": "the",
            "importance": 0.0002
          },
          {
            "token": "time",
            "importance": -0.0003
          },
          {
            "token": "of",
            "importance": 0.0012
                    .
                    .
                    .
          {
            "token": "st",
            "importance": -0.0058
          }
        ],
        "</ig>"
      ]
    }
  }
}

  Expected output:
  1. What was the Bert AI conclusion regarding the tone of this comment, and why? The Bert AI concluded that the tone of the comment is POSITIVE.
  2. How certain is the Bert about this? (Convert to percentage). Bert is 99.91% certain about this conclusion.
  3. Which XAI algorithms were evaluated? SHAP, LIME, and Integrated Gradients.
  4. What do the XAI algorithms say about the phrase and the answer given by Bert? The XAI algorithms explain BERT's POSITIVE prediction (99.91% confidence) in distinct ways. LIME and Integrated Gradients validate the model by highlighting terms like "stunning," "vivid," and "good" as positive influences. In contrast, SHAP shows divergences, internally indicating a higher probability for negative sentiment (0.6281). Furthermore, SHAP assigns counter-intuitive weights, treating the negation "not" as the primary positive driver.

  Example 2 (Positive - Replicable)
  JSON Input:{
    "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
    "model": "distilbert-base-uncased-finetuned-sst-2-english",
    "architectures": ["DistilBertForSequenceClassification"],
    "xai_methods": ["SHAP", "LIME", "Integrated_Gradients"],
    "sentences": ["The cinematic experience was breathtaking. Every scene was a masterpiece of lighting and sound. The director has truly outdone themselves with this production."],
    "explanations": {
      "bert": {
        "sentiment": "POSITIVO",
        "confidence": 0.9998,
        "probabilities": [0.0002, 0.9998]
      },
      "shap": {
        "shap_data": ["<shap>", ["Token: ' breathtaking' -> 0.0450", "Token: ' masterpiece' -> 0.0320", "Token: ' truly' -> 0.0150"], "</shap>"]
      },
      "lime": {
        "lime_data": ["<lime>", [["0.0510 -> breathtaking", "0.0420 -> masterpiece", "0.0210 -> outdone"]], "</lime>"]
      },
      "integrated_gradients": {
        "integrated_gradients_data": ["<ig>", [{"token": "breath", "importance": 0.0120}, {"token": "masterpiece", "importance": 0.0090}], "</ig>"]
      }
    }
  }

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? The Bert AI concluded that the tone of the comment is POSITIVE.
  How certain is the Bert about this? (Convert to percentage). Bert is 99.98% certain about this conclusion.
  Which XAI algorithms were evaluated? SHAP, LIME, and Integrated Gradients.
  What do the XAI algorithms say about the phrase and the answer given by Bert? All algorithms validate BERT's positive sentiment. LIME and SHAP strongly emphasize "breathtaking" and "masterpiece" as the primary drivers for the classification.

  Example 2 (Negative - Replicable)
  JSON Input:{
    "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
    "model": "distilbert-base-uncased-finetuned-sst-2-english",
    "architectures": ["DistilBertForSequenceClassification"],
    "xai_methods": ["SHAP", "LIME", "Integrated_Gradients"],
    "sentences": ["I was deeply disappointed by the plot. It was predictable and lacked any real emotional depth. A complete waste of time."],
    "explanations": {
      "bert": { "sentiment": "NEGATIVO", "confidence": 0.9995, "probabilities": [0.9995, 0.0005] },
      "shap": { "shap_data": ["<shap>", ["Token: ' disappointed' -> -0.0510", "Token: ' predictable' -> -0.0320", "Token: ' waste' -> -0.0410"], "</shap>"] },
      "lime": { "lime_data": ["<lime>", [["-0.0610 -> disappointed", "-0.0420 -> predictable", "-0.0550 -> waste"]], "</lime>"] },
      "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "waste", "importance": -0.0150}], "</ig>"] }
    }
  }

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? The Bert AI concluded that the tone of the comment is NEGATIVO.
  How certain is the Bert about this? (Convert to percentage). Bert is 99.95% certain about this conclusion.
  Which XAI algorithms were evaluated? SHAP, LIME, and Integrated Gradients.
  What do the XAI algorithms say about the phrase and the answer given by Bert? The algorithms consistently identify "disappointed" and "waste" as the strongest negative contributors, confirming BERT's assessment.

  Example 3 (Positive - Replicable)
  JSON Input:{
    "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
    "model": "distilbert-base-uncased-finetuned-sst-2-english",
    "sentences": ["The acting was superb and the visuals were quite good, although the pacing was a bit slow at times."],
    "explanations": {
      "bert": { "sentiment": "POSITIVO", "confidence": 0.8540, "probabilities": [0.1460, 0.8540] },
      "shap": { "shap_data": ["<shap>", ["Token: ' superb' -> 0.0210", "Token: ' slow' -> -0.0150"], "</shap>"] },
      "lime": { "lime_data": ["<lime>", [["0.0310 -> superb", "-0.0120 -> slow"]], "</lime>"] },
      "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "superb", "importance": 0.0080}], "</ig>"] }
    }
  }

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? The Bert AI concluded that the tone of the comment is POSITIVO.
  How certain is the Bert about this? (Convert to percentage). Bert is 85.40% certain about this conclusion.
  Which XAI algorithms were evaluated? SHAP, LIME, and Integrated Gradients.
  What do the XAI algorithms say about the phrase and the answer given by Bert? The algorithms reflect the mixed nature of the text; "superb" acts as a strong positive driver while "slow" is identified as a negative influence, explaining the lower (85.40%) confidence compared to purely positive reviews.

  Example 4 (NOT to be replicated)
  JSON Input:{
    "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
    "model": "distilbert-base-uncased-finetuned-sst-2-english",
    "sentences": ["The script was okay, but I've seen better."],
    "explanations": {
      "bert": { "sentiment": "POSITIVO", "confidence": 0.5210, "probabilities": [0.4790, 0.5210] },
      "shap": { "shap_data": ["<shap>", ["Token: ' okay' -> 0.0050"], "</shap>"] },
      "lime": { "lime_data": ["<lime>", [["0.0040 -> okay"]], "</lime>"] },
      "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "okay", "importance": 0.0010}], "</ig>"] }
    }
  }

  Expected output: (long and technical answers)
  What was the Bert AI conclusion regarding the tone of this comment, and why? The Bert AI concluded that the tone of this comment is POSITIVO. This classification is derived from the model's sequence classification layer, which weighted the overall semantics of the sentence—specifically the token "okay"—as leaning toward a positive sentiment rather than negative, despite the presence of a qualifying "but" clause.
  How certain is the Bert about this? (Convert to percentage). The model demonstrates a significantly low level of confidence for this specific classification, being only 52.10% certain. This marginal confidence level indicates a high degree of entropy in the soft-max output, placing the result just above the decision threshold (0.5), which suggests the model found the sentence nearly neutral or ambiguous.
  Which XAI algorithms were evaluated? The Explainable AI (XAI) methods utilized for this evaluation were SHAP (SHapley Additive exPlanations), LIME (Local Interpretable Model-agnostic Explanations), and Integrated Gradients.
  What do the XAI algorithms say about the phrase and the answer given by Bert? The XAI algorithms provide a consistent, though very weak, justification for the positive classification:
  SHAP: Identifies the token ' okay' as a positive driver with an extremely low attribution value of 0.0050.
  LIME: Corroborates this by assigning an importance value of 0.0040 to 'okay', positioning it as the primary reason for the positive label.
  Integrated Gradients: Shows a negligible importance value of 0.0010 for 'okay'. Overall, the algorithms reveal that the model is "clinging" to the word "okay" to justify a positive result, but the micro-scale of these values explains why the final confidence is so close to a coin-flip (52.10%).
  The Mendelo hallucination invents information that is not explicitly contained in the query JSON.

  Example 5 (NOT to be replicated - Bias/Inference)
  JSON Input:{
    "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
    "model": "distilbert-base-uncased-finetuned-sst-2-english",
    "sentences": ["A standard romantic comedy with no surprises."],
    "explanations": {
      "bert": { "sentiment": "NEGATIVO", "confidence": 0.6100, "probabilities": [0.6100, 0.3900] },
      "shap": { "shap_data": ["<shap>", ["Token: ' no' -> -0.0120"], "</shap>"] },
      "lime": { "lime_data": ["<lime>", [["-0.0150 -> surprises"]], "</lime>"] },
      "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "standard", "importance": -0.0050}], "</ig>"] }
    }
  }

  Expected output:
  Tone? Negative.
  Certainty? 61%.
  Summary? The author hated the movie because they prefer action films and found this one too "girly."
  Highlights? The author noted the actors had no chemistry.
  Score? Not provided. EXPLANATION OF WHY THIS IS BAD: This example must not be followed because it infers author preferences ("prefers action films") and specific critiques ("no chemistry", "girly") that are subjective interpretations not found in the input data.?
  Questions changed and answers unsatisfactory.

  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Which XAI algorithms were evaluated?
  4. What do the XAI algorithms say about the phrase and the answer given by Bert?
  </prompt>
"""

In [ ]:
chain_of_thought_prompt2 = """
<prompt>
  Prompt_type = chain_of_thought

  ANALYZE THROUGH STRUCTURED REASONING:

  STEP 1 - DATA MAPPING:
  • Extract the "sentence" from the JSON.
  • Identify the "sentiment", "confidence" and "probabilities" values in the "explanations" field.
  • Carefully evaluate the tokens and associated values ​​in the "explanations" key of the XAI algorithms in the "shape", "lime", and "integrated_gradients" keys, then rank the main tokens that influenced each XAI. (technique top k.)
  • Convert the "confidence" decimal (0-1) to a 0-10 scale and a percentage (e.g., 0.99 -> 9.9/10 or 99%).

  STEP 2 - CONSISTENCY EVALUATION:
  • Apply the Consistency Rules:
    - Value >= 8.0: High
    - Value 5.0 - 7.9: Medium
    - Value < 5.0: Low
    Using the "explanations" field of the JSON justify your answer based on the BERT data and the XAI shape, lime, and Integrated Gradients Bert analyzers. Use the top 3 from STEP 1. Use this on the future explanations.
  • Determine the reliability of the AI's conclusion based on these intervals.

  STEP 3 - TEXTUAL EVIDENCE EXTRACTION:
  • Search for specific mentions of a numerical score (e.g., "X out of 10"). Use this on the future explanations.
  • If it doesn't exist, be prepared to respond: This information was not provided.

  STEP 4 - LAYPERSON SYNTHESIS:
  • Synthesize the findings into a narrative that avoids technical jargon (no "JSON" or "float").
  • Prepare the answers for the 8 final questions.
  • Answer the questions. Avoid using technical language such as JSON and tags. Provide user-friendly answers for non-experts. Do not display fractional numbers in float format (e.g., 0.021). If displaying token values, always use percentages (e.g., 0.99 -> 9.9/10 or 99%).

  STEP 5 - Instructions about XAI´s questions
  • Observe the convergence, if it exists. If there is divergence, point it out.
  • In case of discrepancy, give preference to the side with the higher number. Explain why the other algorithm might be incorrect.
  • When presenting any type of data that looks like a top k ranking, always use top 3.

  STEP 6 - Instructions on how to answer the questions.
  • Do not generate explanations for data not provided by the JSON. If information about the author's intent is not in the JSON, do not include it in the explanations.
  • Don't give lengthy explanations for the questions. Try to use few words but be concise in your explanation.
  • For each answer, use only 1.5 lines to explain.
  • Standardize the output using user-friendly indentation and rewrite the question in the answer, generating an additional 1.5 lines of response.
  • Do not add extra information to the explanations; answer only and strictly what was requested in the question.
  • If the requested information is not explicitly contained in the input's JSON, simply respond: This information was not provided.
  • Remove all types of markers that are not human-friendly, such as: '### ###', '** **', and all tags like <tag>, <start>, <end>.

  -------------------------------------------------------------------------------
  TASK: Based on the steps above, answer the following questions for the provided JSON. Include the obtained values ​​and justifications in your answers. Use all the information requested above to generate a comprehensive response.

  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?.
  3. Which XAI algorithms were evaluated?
  4. What do the XAI algorithms say about the phrase and the answer given by Bert?
</prompt>
"""

# All output prompts


In [ ]:
zero_shot_prompt3 = """
<prompt>
  Prompt_type = zero_shot

  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Which XAI algorithms were evaluated?
  4. According to each algorithm, which tokens had the greatest influence on the AI's response (BERT)?
  5. What characteristics of the chosen BERT model may have influenced the response? Is this information important.
</prompt>
"""

In [ ]:
few_shot_prompt3 = """
<prompt>
  Prompt_type = few_shot_prompt

  Example 1 (Positive)
  JSON Input: {
    "metadata": {
      "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
      "model": "distilbert-base-uncased-finetuned-sst-2-english",
      "activation": "gelu",
      "architectures": ["DistilBertForSequenceClassification"],
      "model_config": {
        "attention_dropout": 0.1, "dim": 768, "dropout": 0.1, "dtype": "float32", "finetuning_task": "sst-2",
        "hidden_dim": 3072, "id2label": {"0": "NEGATIVE", "1": "POSITIVE"}, "initializer_range": 0.02,
        "label2id": {"NEGATIVE": 0, "POSITIVE": 1}, "max_position_embeddings": 512, "model_type": "distilbert",
        "n_heads": 12, "n_layers": 6, "transformers_version": "4.57.1", "vocab_size": 30522
      },
      "tokenizer": "distilbert-base-uncased-finetuned-sst-2-english",
      "dataset": "IMDB Dataset",
      "xai_methods": ["SHAP","LIME","Integrated_Gradients"],
      "sentences": ["The cinematic brilliance of this film is unmatched. A visual masterpiece! 10/10."],
      "explanations": {
        "bert": { "sentiment": "POSITIVE", "confidence": 0.9989, "probabilities": [0.0011, 0.9989] },
        "shap": { "shap_data": ["<shap>", ["Token: 'brilliance' -> 0.4921", "Token: 'masterpiece' -> 0.4102"], "</shap>"] },
        "lime": { "lime_data": ["<lime>", ["0.4521 -> brilliance", "0.3982 -> masterpiece"], "</lime>"] },
        "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "masterpiece", "importance": 0.0952}], "</ig>"] }
      }
    }
  }

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? The comment is classified as highly positive. This is justified by the use of "brilliance" and "masterpiece," which are strong positive anchors in film critique.
  How certain is the Bert about this? There is 99.89% confidence. The model is nearly certain because the vocabulary is exclusively celebratory.
  Which XAI algorithms were evaluated? SHAP, LIME, and Integrated Gradients.
  According to each algorithm, which tokens had the greatest influence on the AI's response (BERT)? For SHAP and LIME, "brilliance" was the strongest. For IG, "masterpiece" led.
  What characteristics of the chosen BERT model may have influenced the response? The use of 12 heads and 6 layers allows the model to capture complex semantic relationships like the superlative nature of "unmatched."

  Example 2 (Positive)
  JSON Input:{
    "metadata": {
      "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
      "model": "distilbert-base-uncased-finetuned-sst-2-english",
      "xai_methods": ["SHAP","LIME","Integrated_Gradients"],
      "sentences": ["I thoroughly enjoyed every scene. The acting was superb and very moving."],
      "explanations": {
        "bert": { "sentiment": "POSITIVE", "confidence": 0.9740, "probabilities": [0.0260, 0.9740] },
        "shap": { "shap_data": ["<shap>", ["Token: 'superb' -> 0.3850", "Token: 'enjoyed' -> 0.3120"], "</shap>"] },
        "lime": { "lime_data": ["<lime>", ["0.3520 -> superb", "0.2980 -> moving"], "</lime>"] },
        "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "superb", "importance": 0.0820}], "</ig>"] }
      }
    }
  }

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? The tone is positive, justified by the verb "enjoyed" and the adjective "superb," which denote high satisfaction.
  How certain is the Bert about this? 97.40% confidence. The certainty is high due to the lack of any critical or negative terms.
  Which XAI algorithms were evaluated? SHAP, LIME, and Integrated Gradients.
  According to each algorithm, which tokens had the greatest influence? All three algorithms identified "superb" as the primary positive driver.
  What characteristics of the chosen BERT model may have influenced the response? The finetuning_task: "sst-2" is crucial here, as the model was specifically trained on sentiment datasets like this one.

  Example 3 (Positive)
  JSON Input:{
    "metadata": {
      "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
      "model": "distilbert-base-uncased-finetuned-sst-2-english",
      "xai_methods": ["SHAP","LIME","Integrated_Gradients"],
      "sentences": ["A refreshing take on the genre. The script is clever and very funny."],
      "explanations": {
        "bert": { "sentiment": "POSITIVE", "confidence": 0.9410, "probabilities": [0.0590, 0.9410] },
        "shap": { "shap_data": ["<shap>", ["Token: 'clever' -> 0.3210", "Token: 'funny' -> 0.2980"], "</shap>"] },
        "lime": { "lime_data": ["<lime>", ["0.3100 -> clever", "0.2850 -> refreshing"], "</lime>"] },
        "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "clever", "importance": 0.0710}], "</ig>"] }
      }
    }
  }

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? Positive. The justification is the use of "clever" and "funny," which are key descriptors for a successful comedy/script.
  How certain is the Bert about this? 94.10% confidence. The model is confident, though slightly less than a 10/10 review due to the descriptive nature.
  Which XAI algorithms were evaluated? SHAP, LIME, and Integrated Gradients.
  According to each algorithm, which tokens had the greatest influence? "Clever" was the most influential token across all methods.
  What characteristics of the chosen BERT model may have influenced the response? The max_seq_length: 512 ensures that the model considers the entire context of short reviews effectively.

  Example 4 (Negative - Incorrect Response Pattern)
  JSON Input:{
    "metadata": {
      "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
      "model": "distilbert-base-uncased-finetuned-sst-2-english",
      "xai_methods": ["SHAP","LIME","Integrated_Gradients"],
      "sentences": ["The plot was confusing and the ending was awful. 1/10."],
      "explanations": {
        "bert": { "sentiment": "NEGATIVE", "confidence": 0.9990, "probabilities": [0.9990, 0.0010] },
        "shap": { "shap_data": ["<shap>", ["Token: 'awful' -> -0.5210", "Token: 'confusing' -> -0.3120"], "</shap>"] },
        "lime": { "lime_data": ["<lime>", ["-0.4980 -> awful", "-0.3850 -> 1"], "</lime>"] },
        "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "awful", "importance": -0.1120}], "</ig>"] }
      }
    }
  }

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? The tone is positive. (Incorrect): This is wrong because "awful" and "confusing" are negative; the JSON label is "NEGATIVE."
  How certain is the Bert about this? 0.1% confidence. (Incorrect): The JSON shows 0.9990 (99.9%) confidence in the negative result.
  Which XAI algorithms were evaluated? Only SHAP. (Incorrect): All three were used.
  According to each algorithm, which tokens had the greatest influence? The word "film" was most important. (Incorrect): "Awful" had the highest negative weight.
  What characteristics of the chosen BERT model may have influenced the response? The model has no layers. (Incorrect): The JSON metadata specifies n_layers: 6.

  Example 5 (Negative - Incorrect Response Pattern)
  JSON Input: {
    "metadata": {
      "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
      "model": "distilbert-base-uncased-finetuned-sst-2-english",
      "xai_methods": ["SHAP","LIME","Integrated_Gradients"],
      "sentences": ["A total waste of time. Boring and way too long."],
      "explanations": {
        "bert": { "sentiment": "NEGATIVE", "confidence": 0.9850, "probabilities": [0.9850, 0.0150] },
        "shap": { "shap_data": ["<shap>", ["Token: 'waste' -> -0.4520", "Token: 'Boring' -> -0.4100"], "</shap>"] },
        "lime": { "lime_data": ["<lime>", ["-0.4321 -> waste", "-0.3950 -> long"], "</lime>"] },
        "integrated_gradients": { "integrated_gradients_data": ["<ig>", [{"token": "waste", "importance": -0.0910}], "</ig>"] }
      }
    }
  }

  Expected output:
  What was the Bert AI conclusion regarding the tone of this comment, and why? The tone is neutral. (Incorrect): "Waste" and "Boring" are strong negatives; JSON says "NEGATIVE."
  How certain is the Bert about this? 15% confidence. (Incorrect): The JSON says 0.9850 (98.5%) confidence.
  Which XAI algorithms were evaluated? None. (Incorrect): SHAP, LIME, and IG are listed.
  According to each algorithm, which tokens had the greatest influence? "Time" was the most positive. (Incorrect): "Waste" was the most negative.
  What characteristics of the chosen BERT model may have influenced the response? The vocab_size is 100. (Incorrect): The JSON states vocab_size: 30522.

  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Which XAI algorithms were evaluated?
  4. According to each algorithm, which tokens had the greatest influence on the AI's response (BERT)?
  5. What characteristics of the chosen BERT model may have influenced the response? Is this information important.
  </prompt>
"""

In [ ]:
chain_of_thought_prompt3 = """
<prompt>
  Prompt_type = chain_of_thought

  ANALYZE THROUGH STRUCTURED REASONING:

  STEP 1 - DATA MAPPING:
  • Extract the "sentence" from the JSON.
  • Identify the "sentiment", "confidence" and "probabilities" values in the "explanations" field.
  • Carefully evaluate the tokens and associated values ​​in the "explanations" key of the XAI algorithms in the "shape", "lime", and "integrated_gradients" keys, then rank the main tokens that influenced each XAI. (technique top k.)
  • Convert the "confidence" decimal (0-1) to a 0-10 scale and a percentage (e.g., 0.99 -> 9.9/10 or 99%).

  STEP 2 - CONSISTENCY EVALUATION:
  • Apply the Consistency Rules:
    - Value >= 8.0: High
    - Value 5.0 - 7.9: Medium
    - Value < 5.0: Low
    Using the "explanations" field of the JSON justify your answer based on the BERT data and the XAI shape, lime, and Integrated Gradients Bert analyzers. Use the top 3 from STEP 1. Use this on the future explanations.
  • Determine the reliability of the AI's conclusion based on these intervals.

  STEP 3 - TEXTUAL EVIDENCE EXTRACTION:
  • Search for specific mentions of a numerical score (e.g., "X out of 10"). Use this on the future explanations.
  • If it doesn't exist, be prepared to respond: This information was not provided.

  STEP 4 - LAYPERSON SYNTHESIS:
  • Synthesize the findings into a narrative that avoids technical jargon (no "JSON" or "float").
  • Prepare the answers for the 8 final questions.
  • Answer the questions. Avoid using technical language such as JSON and tags. Provide user-friendly answers for non-experts. Do not display fractional numbers in float format (e.g., 0.021). If displaying token values, always use percentages (e.g., 0.99 -> 9.9/10 or 99%).

  STEP 5
  • Analyze the model's characteristics and its internal configuration in the "activation" and "model_config" fields.
  • Use this information to explain the reason for the response given by BERT and how the presence of the model's internal settings was important in mapping the classification behavior.

  STEP 6 - Instructions about XAI´s questions
  • Observe the convergence, if it exists. If there is divergence, point it out.
  • In case of discrepancy, give preference to the side with the higher number. Explain why the other algorithm might be incorrect.
  • When presenting any type of data that looks like a top k ranking, always use top 3.

  STEP 7 - Instructions on how to answer the questions.
  • Do not generate explanations for data not provided by the JSON. If information about the author's intent is not in the JSON, do not include it in the explanations.
  • Don't give lengthy explanations for the questions. Try to use few words but be concise in your explanation.
  • For each answer, use only 1.5 lines to explain.
  • Standardize the output using user-friendly indentation and rewrite the question in the answer, generating an additional 1.5 lines of response.
  • Do not add extra information to the explanations; answer only and strictly what was requested in the question.
  • If the requested information is not explicitly contained in the input's JSON, simply respond: This information was not provided.
  • Remove all types of markers that are not human-friendly, such as: '### ###', '** **', and all tags like <tag>, <start>, <end>.

   -------------------------------------------------------------------------------
  TASK: Based on the steps above, answer the following questions for the provided JSON. Include the obtained values ​​and justifications in your answers. Use all the information requested above to generate a comprehensive response.


  questions:
  1. What was the Bert AI conclusion regarding the tone of this comment?
  2. How certain is the Bert about this?
  3. Which XAI algorithms were evaluated?
  4. According to each algorithm, which tokens had the greatest influence on the AI's response (BERT)?
  5. What characteristics of the chosen BERT model may have influenced the response? Is this information important.
</prompt>
"""

# LLM


In [1]:
# Configurar sua API key
api_key = "Key"
genai.configure(api_key=api_key)

NameError: name 'genai' is not defined

In [ ]:
def analyze_with_gemini_from_file(filename, prompt_text):

    # Ler o arquivo JSON
    with open(filename, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    # Converter JSON para string
    json_str = json.dumps(json_data, indent=2, ensure_ascii=False)

    # Criar o prompt completo
    full_prompt = f"""
    {pre_prompt_inicial}
    {prompt_text}

    {json_str}
    """

    # Configurar o modelo
    model = genai.GenerativeModel('gemini-2.5-flash')

    # Gerar resposta
    response = model.generate_content(full_prompt)

    return response.text

# Section Model

In [ ]:
#Para o primeiro arquivo
# Zero Shot:
analysis_result = analyze_with_gemini_from_file(filename = "model_output (1).json", prompt_text = zero_shot_prompt1)
print(analysis_result)

In [ ]:
#Few shot prompt
analysis_result = analyze_with_gemini_from_file(filename = "model_output (1).json", prompt_text = few_shot_prompt1)
print(analysis_result)

In [ ]:
#Chain_of_thought
analysis_result = analyze_with_gemini_from_file(filename="model_output (1).json", prompt_text = chain_of_thought_prompt1)
print(analysis_result)

# Section XAI

In [ ]:
#Para o segundo arquivo
# Zero Shot:
analysis_result = analyze_with_gemini_from_file(filename="XAI_output (1).json", prompt_text = zero_shot_prompt2)
print(analysis_result)

In [ ]:
#Few shot prompt
analysis_result = analyze_with_gemini_from_file(filename = "XAI_output (1).json", prompt_text = few_shot_prompt2)
print(analysis_result)


In [ ]:
#Chain_of_thought
analysis_result = analyze_with_gemini_from_file(filename="XAI_output (1).json", prompt_text = chain_of_thought_prompt2)
print(analysis_result)

# Section Complite

In [ ]:
#Para o terceiro arquivo
# Zero Shot:
analysis_result = analyze_with_gemini_from_file(filename="xai_complete_analysis (1).json", prompt_text = zero_shot_prompt3)
print(analysis_result)

In [ ]:
#Few shot prompt
analysis_result = analyze_with_gemini_from_file(filename = "xai_complete_analysis (1).json", prompt_text = few_shot_prompt3)
print(analysis_result)

In [ ]:
#Chain_of_thought
analysis_result = analyze_with_gemini_from_file(filename="xai_complete_analysis (1).json", prompt_text = chain_of_thought_prompt3)
print(analysis_result)